# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Colab starts in /content; clone the public fork so scripts and configs are available.
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/K4-Track-4-Day10-VisionLanguageModel")
REPO_URL = "https://github.com/Duytd26/K4-Track-4-Day10-VisionLanguageModel.git"
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
%cd /content/K4-Track-4-Day10-VisionLanguageModel


In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Run the assignment integrity checks before downloading data.
!python -m pytest -q


In [ ]:
# Mount Drive so adapters and checkpoints survive Colab restarts.
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_RUNS = Path("/content/drive/MyDrive/day10-vlm/runs")
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
print(f"Artifacts will be backed up under: {DRIVE_RUNS}")


In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# Verify the fixed session split, compare counts, and inspect six labeled training images.
import json
from collections import Counter
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display
from scripts.labcommon import read_jsonl

DATA_ROOT = Path("data")
SPLIT_DIR = DATA_ROOT / "splits"
with (SPLIT_DIR / "meta.json").open(encoding="utf-8") as f:
    split_meta = json.load(f)
dataset_version = split_meta["dataset_version"]
assert dataset_version.startswith("d10-"), dataset_version

expected = {
    "train": {"images": 1422, "boxes": {"pothole": 1074, "crack": 1750, "manhole": 691}},
    "val":   {"images": 276,  "boxes": {"pothole": 66,   "crack": 470,  "manhole": 103}},
    "test":  {"images": 311,  "boxes": {"pothole": 121,  "crack": 299,  "manhole": 163}},
}
split_rows = {s: read_jsonl(SPLIT_DIR / f"{s}.jsonl") for s in ("train", "val", "test")}
split_sessions = {s: {r["session"] for r in rows} for s, rows in split_rows.items()}
for s, rows in split_rows.items():
    image_count = len(rows)
    box_counts = Counter(o["label"] for r in rows for o in r["objects"])
    assert image_count == expected[s]["images"], (s, image_count)
    assert dict(box_counts) == expected[s]["boxes"], (s, dict(box_counts))
    assert split_meta["counts"][s]["images"] == image_count
    assert split_meta["counts"][s]["boxes"] == dict(box_counts)
    assert all(split_meta["groups"].get(session) == s for session in split_sessions[s])
for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
    overlap = split_sessions[left] & split_sessions[right]
    assert not overlap, f"Session leakage between {left} and {right}: {sorted(overlap)[:5]}"

print(f"dataset_version: {dataset_version}")
print("split  images  pothole  crack  manhole  sessions")
for s, rows in split_rows.items():
    c = split_meta["counts"][s]["boxes"]
    print(f"{s:5}  {len(rows):6}  {c['pothole']:7}  {c['crack']:5}  {c['manhole']:7}  {len(split_sessions[s]):8}")
print("Session sets are disjoint across train/val/test.")

train_samples = split_rows["train"]
colors = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}
chosen, used_sessions = [], set()
for row in sorted(train_samples, key=lambda r: r["image_id"]):
    if row["session"] not in used_sessions and row["objects"]:
        chosen.append(row)
        used_sessions.add(row["session"])
    if len(chosen) == 6:
        break
assert len(chosen) == 6, "Need six labeled training images from distinct sessions."

tile_w, tile_h, caption_h = 320, 180, 28
canvas = Image.new("RGB", (tile_w * 3, (tile_h + caption_h) * 2), "white")
for i, row in enumerate(chosen):
    img = Image.open(DATA_ROOT / row["image_path"]).convert("RGB")
    draw = ImageDraw.Draw(img)
    W, H = img.size
    assert (W, H) == (row["width"], row["height"])
    for obj in row["objects"]:
        xc, yc, bw, bh = obj["yolo"]
        x1 = max(0, min(W - 1, round((xc - bw / 2) * W)))
        y1 = max(0, min(H - 1, round((yc - bh / 2) * H)))
        x2 = max(0, min(W - 1, round((xc + bw / 2) * W)))
        y2 = max(0, min(H - 1, round((yc + bh / 2) * H)))
        color = colors[obj["label"]]
        draw.rectangle((x1, y1, x2, y2), outline=color, width=3)
        draw.text((x1, max(0, y1 - 12)), obj["label"], fill=color, stroke_width=1, stroke_fill="black")
    img = img.resize((tile_w, tile_h))
    x = (i % 3) * tile_w
    y = (i // 3) * (tile_h + caption_h)
    canvas.paste(img, (x, y))
    caption = f"{row['image_id']} | {row['session']}"
    ImageDraw.Draw(canvas).text((x + 4, y + tile_h + 4), caption, fill="black")
display(canvas)


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# YOUR CODE HERE: viết yolo_to_box() và objects_to_answer() (GUIDE bước 2)

## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9